[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week12.ipynb)

# Week 12: Representation Learning
**Introduction to Deep Learning (HIT)** &middot; Part III: Architectures & Representation Learning

Autoencoders and latent representations; contrastive and self-supervised methods.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Train an autoencoder and a contrastive embedding.
- Probe and interpret a learned latent space.
- Reason about what makes a representation useful.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. Train an autoencoder on MNIST
Compress to a small bottleneck, then reconstruct (downloads ~10 MB).

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
dl = DataLoader(datasets.MNIST("./data", train=True, download=True, transform=transforms.ToTensor()),
                batch_size=256, shuffle=True)
ae = nn.Sequential(nn.Flatten(), nn.Linear(784, 64), nn.ReLU(), nn.Linear(64, 16), nn.ReLU(),   # encoder
                   nn.Linear(16, 64), nn.ReLU(), nn.Linear(64, 784), nn.Sigmoid()).to(device)   # decoder
opt = torch.optim.Adam(ae.parameters(), 1e-3); f = nn.MSELoss()
for epoch in range(2):
    for xb, _ in dl:
        xb = xb.to(device); opt.zero_grad(); f(ae(xb).view_as(xb), xb).backward(); opt.step()
    print(f"epoch {epoch}: reconstruction MSE {f(ae(xb).view_as(xb), xb).item():.4f}")

## 2. Reconstructions
Top row: originals; bottom row: the autoencoder's reconstruction through a 16-d bottleneck.

In [ ]:
xb, _ = next(iter(dl)); xb = xb.to(device); rec = ae(xb).view_as(xb).detach().cpu()
fig, ax = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    ax[0, i].imshow(xb[i, 0].cpu(), cmap="gray"); ax[0, i].axis("off")
    ax[1, i].imshow(rec[i, 0], cmap="gray"); ax[1, i].axis("off")
plt.show()

## 3. Latent interpolation
Walk in pixel space between two digits, the idea behind interpolating in a learned latent space.

In [ ]:
a, b = xb[0].cpu(), xb[1].cpu()
fig, ax = plt.subplots(1, 6, figsize=(9, 2))
for i, alpha in enumerate(torch.linspace(0, 1, 6)):
    ax[i].imshow(((1 - alpha) * a + alpha * b)[0], cmap="gray"); ax[i].axis("off")
plt.show()

## 4. The contrastive idea
Two augmented views of one image form a positive pair; everything else is negative.

In [ ]:
from torchvision import transforms
aug = transforms.Compose([transforms.RandomResizedCrop(28, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip()])
img = xb[0].cpu()
fig, ax = plt.subplots(1, 3, figsize=(6, 2))
for a_, t, title in zip(ax, [img, aug(img), aug(img)], ["original", "view 1", "view 2"]):
    a_.imshow(t[0], cmap="gray"); a_.set_title(title); a_.axis("off")
plt.show()

**Discuss:** the augmentation policy defines what 'similar' means. What would the model learn if the only augmentation were a color shift?

---
Student materials for this week: the lab handout (`labs/week12.html`) and the curated references (`references/week12.html`) in the course site.